# EXP031: Oracle MFI + MLE (Python) with cumulative Fisher information

通常版の `Test_MFI_MLE_on_the_simulated_bank.ipynb` をベースに、項目選択時の Fisher 情報量を各受検者の真の theta で計算するoracle版です。各stepでは、真の theta における Fisher 情報量が最大の未出題項目を選択します。

回答生成、回答後のMLE推定、Bias・RMSE・MAE・相関の計算、乱数seedは通常版と共通です。累積報酬も通常版と同じく、選択項目の真の theta における Fisher 情報量の和です。通常MFIの結果を上書きしないよう、出力名には `MFI_oracle_MLE` を使用します。

In [25]:
# -*- coding: utf-8 -*-
from dataclasses import dataclass
from pathlib import Path
from typing import Any, cast

import numpy as np
import pandas as pd
from scipy.optimize import minimize_scalar


def find_project_root():
    candidates = []
    if "__file__" in globals():
        script_dir = Path(__file__).resolve().parent
        candidates.extend([script_dir, *script_dir.parents])

    cwd = Path.cwd().resolve()
    candidates.extend(
        [
            cwd,
            *cwd.parents,
            cwd / "Grad_Research",
            cwd
            / "Adaptive-Testing-Based-on-Reinforcement-Learning-Considering-Response-History",
            Path("/content/Grad_Research"),
            Path(
                "/content/Adaptive-Testing-Based-on-Reinforcement-Learning-Considering-Response-History"
            ),
            Path("/content/drive/MyDrive/Grad_Research"),
            Path(
                "/content/drive/MyDrive/Adaptive-Testing-Based-on-Reinforcement-Learning-Considering-Response-History"
            ),
            Path("/content/drive/MyDrive/Colab Notebooks/Grad_Research"),
            Path(
                "/content/drive/MyDrive/Colab Notebooks/Adaptive-Testing-Based-on-Reinforcement-Learning-Considering-Response-History"
            ),
        ]
    )

    for root in candidates:
        if (root / "data").is_dir():
            return root

    try:
        from google.colab import drive

        drive.mount("/content/drive")
    except Exception:
        pass

    for root in candidates:
        if (root / "data").is_dir():
            return root

    raise FileNotFoundError(
        "Could not find the project root. "
        "In Colab, place the repository at MyDrive/Grad_Research or /content/Grad_Research."
    )


ROOT = find_project_root()
RESULTS_DIR = ROOT / "EXP031" / "results"

print(f"Project root: {ROOT}")
print(f"Results dir : {RESULTS_DIR}")

Project root: /Users/itsuki/Adaptive-Testing-Based-on-Reinforcement-Learning-Considering-Response-History
Results dir : /Users/itsuki/Adaptive-Testing-Based-on-Reinforcement-Learning-Considering-Response-History/EXP031/results


In [26]:
@dataclass
class Config:
    test_length: int = 40

    # Bank / evaluation data
    bank_type: str = "uncor"  # 'uncor' | 'cor'
    bank_id: int = 1
    n_items: int = 500
    testing_size: int = 0  # 0: use all theta values
    theta_csv: str = ""  # empty: data/theta_true/theta_true_{bank_id}.csv

    # Reproducibility / output
    seed: int = 20260430
    output_suffix: str = "_python"

In [27]:
# These functions are aligned with the EXP031 DQN notebook.
def RESPOND(item_para, theta, D=1):
    a = item_para[:, 0]
    b = item_para[:, 1]
    c = item_para[:, 2]
    p = (1 - c) / (1 + np.exp(-D * a * (theta - b))) + c
    return (np.random.random(size=p.shape) <= p).astype(int)


def FI(item_para, theta, D=1):
    a = item_para[:, 0]
    b = item_para[:, 1]
    c = item_para[:, 2]
    return (
        D**2
        * a**2
        * (1 - c)
        / (c + np.exp(D * a * (theta - b)))
        / (1 + np.exp(-D * a * (theta - b))) ** 2
    )


def MLE(item_paras, resp, D=1):
    a = item_paras[:, 0]
    b = item_paras[:, 1]
    c = item_paras[:, 2]

    def mins_likelihood(x):
        logl = 0
        for i in range(len(resp)):
            p = (1 - c[i]) / (1 + np.exp(-D * a[i] * (x - b[i]))) + c[i]
            p = np.clip(p, 1e-10, 1 - 1e-10)
            logl -= resp[i] * np.log(p) + (1 - resp[i]) * np.log(1 - p)
        return logl

    result = cast(
        Any, minimize_scalar(mins_likelihood, bounds=(-4, 4), method="bounded")
    )
    return np.array(result.x).reshape(
        1,
    )


def MLE_TEST(item_paras, resp, D=1):
    def mins_likelihood(x):
        logl = 0
        for i in range(resp_i.shape[0]):
            p = (1 - c[i]) / (1 + np.exp(-D * a[i] * (x - b[i]))) + c[i]
            p = np.clip(p, 1e-10, 1 - 1e-10)
            logl -= resp_i[i] * np.log(p) + (1 - resp_i[i]) * np.log(1 - p)
        return logl

    theta = np.zeros(resp.shape[1])
    for i in range(resp.shape[1]):
        resp_i = resp[:, i]
        a = item_paras[:, i, 0]
        b = item_paras[:, i, 1]
        c = item_paras[:, i, 2]
        result = cast(
            Any, minimize_scalar(mins_likelihood, bounds=(-4, 4), method="bounded")
        )
        theta[i] = result.x
    return np.expand_dims(theta, axis=0)

In [28]:
def choose_mfi_oracle(item_bank, theta_true, item_ids):
    """Choose each subject's unadministered item using FI at true theta."""
    information = np.vstack([FI(item_bank, theta) for theta in theta_true])

    if item_ids.shape[0] > 0:
        subject_indices = np.arange(len(theta_true))[:, None]
        information[subject_indices, item_ids.T] = -np.inf

    return information.argmax(axis=1).astype(np.int64)


def estimate_theta_mle(item_bank, item_ids, responses, current_theta):
    testing_size = responses.shape[1]
    theta_hat = np.zeros(testing_size)
    idx_full = np.sum(responses, axis=0) == responses.shape[0]
    idx_zero = np.sum(responses, axis=0) == 0
    idx_norm = ~(idx_full | idx_zero)

    theta_hat[idx_full] = (
        current_theta[idx_full] + (item_bank[:, 1].max() - current_theta[idx_full]) / 2
    )
    theta_hat[idx_zero] = (
        current_theta[idx_zero] - (current_theta[idx_zero] - item_bank[:, 1].min()) / 2
    )
    if np.any(idx_norm):
        theta_hat[idx_norm] = np.squeeze(
            MLE_TEST(
                item_bank[item_ids[:, idx_norm]],
                responses[:, idx_norm],
            )
        )
    return theta_hat


def summarize_steps(theta_true, theta_history, cum_reward_history):
    rows = []
    theta_true_sd = np.std(theta_true, ddof=1)

    for step, (theta_est, cum_reward) in enumerate(
        zip(theta_history, cum_reward_history), start=1
    ):
        bias = theta_est - theta_true
        theta_est_sd = np.std(theta_est, ddof=1)
        correlation = (
            np.nan
            if theta_true_sd == 0 or theta_est_sd == 0
            else np.corrcoef(theta_true, theta_est)[0, 1]
        )
        rows.append(
            {
                "step": step,
                "Bias": np.mean(bias),
                "RMSE": np.sqrt(np.mean(bias**2)),
                "MAE": np.mean(np.abs(bias)),
                "r": correlation,
                "CumReward": float(np.mean(cum_reward)),
            }
        )

    return pd.DataFrame(rows)


def run_mfi(cfg, item_bank, theta_true):
    # Use the same NumPy RNG and subject-vectorized order as DQN TEST.
    np.random.seed(cfg.seed)
    testing_size = len(theta_true)

    theta_current = np.random.rand(testing_size) - 0.5
    item_ids = np.empty((0, testing_size), dtype=np.int64)
    responses = np.empty((0, testing_size), dtype=np.int64)
    theta_history = np.empty((0, testing_size), dtype=float)
    cum_reward = np.zeros(testing_size, dtype=float)
    cum_reward_history = np.empty((0, testing_size), dtype=float)

    for step in range(cfg.test_length):
        selected = choose_mfi_oracle(item_bank, theta_true, item_ids)
        reward = FI(item_bank[selected], theta_true)
        cum_reward += reward
        step_responses = RESPOND(item_bank[selected], theta_true)

        item_ids = np.concatenate((item_ids, selected[np.newaxis, :]))
        responses = np.concatenate((responses, step_responses[np.newaxis, :]))
        theta_current = estimate_theta_mle(
            item_bank, item_ids, responses, theta_current
        )

        theta_history = np.concatenate((theta_history, theta_current[np.newaxis, :]))
        cum_reward_history = np.concatenate(
            (cum_reward_history, cum_reward[np.newaxis, :])
        )
        bias = theta_current - theta_true
        print(
            "step {:g}, bias {:.3f}, rmse {:.3f}, mae {:.3f}, cum_reward {:.3f}".format(
                step + 1,
                np.mean(bias),
                np.sqrt(np.mean(bias**2)),
                np.mean(np.abs(bias)),
                float(np.mean(cum_reward)),
            )
        )

    user_id_col = np.repeat(np.arange(1, testing_size + 1), cfg.test_length)
    step_col = np.tile(np.arange(1, cfg.test_length + 1), testing_size)
    records = pd.DataFrame(
        {
            "userID": user_id_col,
            "step": step_col,
            "itemID": (item_ids + 1).T.reshape(-1),
            "resp": responses.T.reshape(-1),
            "theta_true": np.repeat(theta_true, cfg.test_length),
            "theta_est": theta_history.T.reshape(-1),
            "bias": (theta_history - theta_true).T.reshape(-1),
            "cum_reward": cum_reward_history.T.reshape(-1),
        }
    )
    summary_by_step = summarize_steps(theta_true, theta_history, cum_reward_history)
    return records, summary_by_step

In [29]:
cfg = Config(
    test_length=40,
    bank_type="uncor",
    bank_id=5,
    n_items=500,
    testing_size=0,
    theta_csv="",
    seed=20260430,
    output_suffix="_python",
)

bank_dir = {
    "uncor": ROOT / "data" / "uncorrelated_banks",
    "cor": ROOT / "data" / "correlated_banks",
}.get(cfg.bank_type)
if bank_dir is None:
    raise ValueError("bank_type must be 'uncor' or 'cor'.")

bank_path = bank_dir / f"item_bank_{cfg.bank_type}_{cfg.bank_id}.csv"
item_bank = pd.read_csv(bank_path)[["a", "b", "c"]].to_numpy()[: cfg.n_items]

if cfg.theta_csv:
    theta_path = Path(cfg.theta_csv).expanduser()
    if not theta_path.is_absolute():
        theta_path = ROOT / theta_path
else:
    theta_path = ROOT / "data" / "theta_true" / f"theta_true_{cfg.bank_id}.csv"

theta_true = pd.read_csv(theta_path)["x"].to_numpy()
if not cfg.theta_csv and cfg.testing_size > 0:
    theta_true = theta_true[: cfg.testing_size]

if cfg.test_length > len(item_bank):
    raise ValueError("test_length cannot exceed the number of items in the bank.")
if len(theta_true) < 2:
    raise ValueError("At least two theta values are required to calculate correlation.")

print(f"item bank  : {item_bank.shape} ({bank_path})")
print(f"theta_true : {theta_true.shape} ({theta_path})")
print(f"Config     : {cfg}")

item bank  : (500, 3) (/Users/itsuki/Adaptive-Testing-Based-on-Reinforcement-Learning-Considering-Response-History/data/uncorrelated_banks/item_bank_uncor_5.csv)
theta_true : (5000,) (/Users/itsuki/Adaptive-Testing-Based-on-Reinforcement-Learning-Considering-Response-History/data/theta_true/theta_true_5.csv)
Config     : Config(test_length=40, bank_type='uncor', bank_id=5, n_items=500, testing_size=0, theta_csv='', seed=20260430, output_suffix='_python')


In [30]:
records, summary_by_step = run_mfi(cfg, item_bank, theta_true)

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
stem = f"{cfg.bank_type}_{cfg.bank_id}_MFI_oracle_MLE{cfg.output_suffix}"
records_path = RESULTS_DIR / f"records_{stem}.csv"
summary_path = RESULTS_DIR / f"summary_{stem}.csv"
records.to_csv(records_path, index=False)
summary_by_step.to_csv(summary_path, index=False)

display(summary_by_step.tail(1))
print(f"Saved records to: {records_path}")
print(f"Saved summary to: {summary_path}")

step 1, bias 0.400, rmse 1.653, mae 1.409, cum_reward 0.515
step 2, bias 0.385, rmse 1.730, mae 1.381, cum_reward 0.952
step 3, bias 0.275, rmse 1.687, mae 1.179, cum_reward 1.372
step 4, bias 0.181, rmse 1.504, mae 1.008, cum_reward 1.773
step 5, bias 0.119, rmse 1.327, mae 0.866, cum_reward 2.164
step 6, bias 0.084, rmse 1.129, mae 0.723, cum_reward 2.548
step 7, bias 0.065, rmse 0.982, mae 0.637, cum_reward 2.922
step 8, bias 0.048, rmse 0.890, mae 0.585, cum_reward 3.288
step 9, bias 0.045, rmse 0.783, mae 0.530, cum_reward 3.648
step 10, bias 0.036, rmse 0.702, mae 0.484, cum_reward 4.000
step 11, bias 0.037, rmse 0.641, mae 0.458, cum_reward 4.348
step 12, bias 0.030, rmse 0.593, mae 0.429, cum_reward 4.690
step 13, bias 0.030, rmse 0.565, mae 0.411, cum_reward 5.028
step 14, bias 0.023, rmse 0.520, mae 0.390, cum_reward 5.361
step 15, bias 0.020, rmse 0.500, mae 0.374, cum_reward 5.688
step 16, bias 0.020, rmse 0.473, mae 0.358, cum_reward 6.011
step 17, bias 0.016, rmse 0.454, 

,step,Bias,RMSE,MAE,r,CumReward
39,40,0.00257,0.295117,0.230617,0.958363,13.022928


Saved records to: /Users/itsuki/Adaptive-Testing-Based-on-Reinforcement-Learning-Considering-Response-History/EXP031/results/records_uncor_5_MFI_oracle_MLE_python.csv
Saved summary to: /Users/itsuki/Adaptive-Testing-Based-on-Reinforcement-Learning-Considering-Response-History/EXP031/results/summary_uncor_5_MFI_oracle_MLE_python.csv
